#Set up

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Função do gráfico

In [ ]:
def plot_heatmap_hora_dimensao(
    pivot,
    titulo="Heatmap por dimensão de calendário",
    subtitulo=None,
    colormap="RdBu",
    inverter_escala_heatmap=False,
    ponto_neutro=None,
    zmin=None,
    zmax=None,
    exibir_texto=True,
    texto_template=".2f",
    tamanho_fonte_texto=11,
    x_label="Dimensão temporal",
    y_label="Categoria",
    largura_px=1100,
    altura_px=650,
    tamanho_fonte_titulo=20,
    tamanho_fonte_subtitulo=12,
    tamanho_fonte_eixos=12,
    tamanho_fonte_ticks=10,
    rotacao_xticks=0,
    rotacao_yticks=0,
    formatar_colunas_como_hora=False,
    sufixo_hora="h",
    inverter_eixo_y=True,
    mostrar_colorbar=True,
    colorbar_title="Valor",
    colorbar_tickformat=".2f",
    mostrar_linha_ponto_neutro=True,
    cor_linha_ponto_neutro="black",
    espessura_linha_ponto_neutro=2,
    estilo_linha_ponto_neutro="dash",
    mostrar_texto_ponto_neutro=True,
    texto_ponto_neutro=None,
    tamanho_fonte_ponto_neutro=11,
    cor_fundo="white",
    margem_esquerda=80,
    margem_direita=80,
    margem_superior=100,
    margem_inferior=70,
    exibir_figura=True,
):
    import pandas as pd
    import numpy as np
    import plotly.graph_objects as go

    # ==========================================================
    # 1. Validações iniciais
    # ==========================================================
    # Garante que o input é um DataFrame válido
    if not isinstance(pivot, pd.DataFrame):
        raise TypeError("O parâmetro 'pivot' deve ser um pandas.DataFrame.")

    # Evita rodar gráfico com DataFrame vazio
    if pivot.empty:
        raise ValueError("O DataFrame 'pivot' está vazio.")

    # Evita caso degenerado (sem linhas ou colunas)
    if pivot.shape[0] == 0 or pivot.shape[1] == 0:
        raise ValueError("O DataFrame 'pivot' precisa ter linhas e colunas válidas.")

    # ==========================================================
    # 2. Preparação dos dados
    # ==========================================================
    # Cria cópia para não alterar o DataFrame original
    pivot_plot = pivot.copy()

    # Garante que todas as colunas sejam numéricas
    # Isso evita erro no heatmap e força coerção de strings inválidas para NaN
    for col in pivot_plot.columns:
        pivot_plot[col] = pd.to_numeric(pivot_plot[col], errors="coerce")

    # Verifica se todos os valores viraram NaN (caso extremo)
    if pivot_plot.isna().all().all():
        raise ValueError("Todos os valores do DataFrame são inválidos ou não numéricos.")

    # ---------------------------
    # Construção do eixo X
    # ---------------------------
    # Permite formatar como hora (ex: 8 -> 08h)
    if formatar_colunas_como_hora:
        x_vals = []
        for valor in pivot_plot.columns:
            try:
                hora = int(valor)
                x_vals.append(f"{hora:02d}{sufixo_hora}")
            except Exception:
                # fallback se não for possível converter
                x_vals.append(str(valor))
    else:
        # Apenas converte para string
        x_vals = [str(col) for col in pivot_plot.columns]

    # ---------------------------
    # Construção do eixo Y
    # ---------------------------
    # Converte index para string para evitar problemas no Plotly
    y_vals = [str(idx) for idx in pivot_plot.index]

    # ---------------------------
    # Matriz Z (valores do heatmap)
    # ---------------------------
    z_vals = pivot_plot.values.astype(float)

    # ---------------------------
    # Texto dentro das células
    # ---------------------------
    if exibir_texto:
        # Cria matriz de textos com mesmo shape do heatmap
        text_vals = np.empty_like(z_vals, dtype=object)

        for i in range(z_vals.shape[0]):
            for j in range(z_vals.shape[1]):
                valor = z_vals[i, j]

                # Evita mostrar "nan" visualmente
                if pd.isna(valor):
                    text_vals[i, j] = ""
                else:
                    text_vals[i, j] = format(valor, texto_template)

        texttemplate = "%{text}"
    else:
        text_vals = None
        texttemplate = None

    # ==========================================================
    # 3. Escala de cor
    # ==========================================================
    # Identifica limites reais dos dados
    valor_min = np.nanmin(z_vals)
    valor_max = np.nanmax(z_vals)

    # Usa limites automáticos se não forem definidos
    if zmin is None:
        zmin = valor_min

    if zmax is None:
        zmax = valor_max

    # Proteção contra caso em que todos os valores são iguais
    # Evita erro no Plotly
    if zmin == zmax:
        zmin = zmin - 1
        zmax = zmax + 1

    # ---------------------------
    # Inversão da escala de cor
    # ---------------------------
    # Permite alternar visualmente o significado das cores
    if inverter_escala_heatmap:
        if isinstance(colormap, str) and not colormap.endswith("_r"):
            colormap = f"{colormap}_r"
    else:
        if isinstance(colormap, str) and colormap.endswith("_r"):
            colormap = colormap[:-2]

    # ==========================================================
    # 4. Montagem do heatmap
    # ==========================================================
    # Parâmetros base do heatmap
    heatmap_kwargs = dict(
        z=z_vals,
        x=x_vals,
        y=y_vals,
        colorscale=colormap,
        zmin=zmin,
        zmax=zmax,
        showscale=mostrar_colorbar,
        colorbar=dict(
            title=colorbar_title,
            tickformat=colorbar_tickformat
        ),
        # Hover padrão (tooltip)
        hovertemplate=(
            f"{y_label}: %{{y}}<br>"
            f"{x_label}: %{{x}}<br>"
            "Valor: %{z}<extra></extra>"
        )
    )

    # Define ponto médio da escala (ex: zero)
    if ponto_neutro is not None:
        heatmap_kwargs["zmid"] = ponto_neutro

    # Adiciona textos nas células se ativado
    if exibir_texto:
        heatmap_kwargs["text"] = text_vals
        heatmap_kwargs["texttemplate"] = texttemplate
        heatmap_kwargs["textfont"] = dict(size=tamanho_fonte_texto)

    # Criação do gráfico
    fig = go.Figure(data=go.Heatmap(**heatmap_kwargs))

    # ==========================================================
    # 5. Título
    # ==========================================================
    # Monta título com ou sem subtítulo
    if subtitulo:
        titulo_final = (
            f"<b>{titulo}</b>"
            f"<br><span style='font-size:{tamanho_fonte_subtitulo}px'>{subtitulo}</span>"
        )
    else:
        titulo_final = f"<b>{titulo}</b>"

    # ==========================================================
    # 6. Layout
    # ==========================================================
    fig.update_layout(
        title=dict(
            text=titulo_final,
            x=0.5,
            xanchor="center",
            font=dict(size=tamanho_fonte_titulo)
        ),
        width=largura_px,
        height=altura_px,
        paper_bgcolor=cor_fundo,
        plot_bgcolor=cor_fundo,
        margin=dict(
            l=margem_esquerda,
            r=margem_direita,
            t=margem_superior,
            b=margem_inferior
        ),
        xaxis=dict(
            title=x_label,
            tickangle=rotacao_xticks,
            title_font=dict(size=tamanho_fonte_eixos),
            tickfont=dict(size=tamanho_fonte_ticks),
            side="bottom"
        ),
        yaxis=dict(
            title=y_label,
            tickangle=rotacao_yticks,
            title_font=dict(size=tamanho_fonte_eixos),
            tickfont=dict(size=tamanho_fonte_ticks),
            autorange="reversed" if inverter_eixo_y else True
        )
    )

    # ==========================================================
    # 7. Linha de referência do ponto neutro
    # ==========================================================
    # Como Plotly não permite desenhar direto na colorbar,
    # fazemos uma aproximação visual no "paper"
    if (
        mostrar_colorbar
        and ponto_neutro is not None
        and mostrar_linha_ponto_neutro
        and zmax > zmin
    ):
        # Calcula posição relativa do ponto neutro na escala
        pos_relativa = (ponto_neutro - zmin) / (zmax - zmin)
        pos_relativa = max(0, min(1, pos_relativa))

        # Ajuste visual da posição vertical
        y0_paper = 0.15 + pos_relativa * 0.70

        # Linha ao lado da colorbar
        fig.add_shape(
            type="line",
            xref="paper",
            yref="paper",
            x0=1.02,
            x1=1.07,
            y0=y0_paper,
            y1=y0_paper,
            line=dict(
                color=cor_linha_ponto_neutro,
                width=espessura_linha_ponto_neutro,
                dash=estilo_linha_ponto_neutro
            )
        )

        # Texto do ponto neutro
        if mostrar_texto_ponto_neutro:
            texto_final = str(ponto_neutro) if texto_ponto_neutro is None else texto_ponto_neutro

            fig.add_annotation(
                xref="paper",
                yref="paper",
                x=1.075,
                y=y0_paper,
                text=texto_final,
                showarrow=False,
                xanchor="left",
                yanchor="middle",
                font=dict(
                    size=tamanho_fonte_ponto_neutro,
                    color=cor_linha_ponto_neutro
                )
            )

    # ==========================================================
    # 8. Exibição
    # ==========================================================
    if exibir_figura:
        fig.show()

    return fig, pivot_plot

# Dados de demostração

In [ ]:
pivot_teste = pd.DataFrame(
    {
        h: [
            np.round(np.random.normal(loc=3.5 + (h >= 8 and h <= 11) * 1.5 + (h >= 13 and h <= 17) * 0.8, scale=0.5), 1),
            np.round(np.random.normal(loc=3.8 + (h >= 8 and h <= 11) * 1.2 + (h >= 13 and h <= 17) * 0.6, scale=0.5), 1),
            np.round(np.random.normal(loc=4.5 + (h >= 8 and h <= 11) * 1.8 + (h >= 13 and h <= 17) * 1.0, scale=0.6), 1),
            np.round(np.random.normal(loc=4.0 + (h >= 8 and h <= 11) * 1.0 + (h >= 13 and h <= 17) * 0.7, scale=0.5), 1),
        ]
        for h in range(24)
    },
    index=["Clínica Médica", "Pediatria", "Ginecologia", "Cardiologia"]
)

In [ ]:
pivot_teste.head(3)

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
Clínica Médica,3.9,3.1,4.0,3.8,3.8,3.5,3.3,4.2,4.7,4.9,...,3.9,4.5,4.8,4.9,3.2,3.4,3.1,2.6,3.1,3.6
Pediatria,4.4,3.3,2.6,4.2,3.8,2.9,4.1,4.6,5.1,5.1,...,4.0,4.3,4.5,4.6,3.5,3.6,4.5,3.9,3.8,4.1
Ginecologia,4.5,5.6,5.8,5.2,3.5,3.8,4.8,4.4,5.5,7.0,...,5.2,4.8,5.4,6.1,4.8,3.4,3.6,4.6,4.8,4.0


# Plotagem

In [ ]:
fig, pivot_plot = plot_heatmap_hora_dimensao(
    pivot=pivot_teste,
    colormap="RdBu",
    inverter_escala_heatmap=True,
    titulo="Atendimentos por especialidade ao longo do dia",
    subtitulo="Referência centrada em 4 atendimentos por profissional/hora",
    ponto_neutro=4,
    x_label="Hora do dia",
    y_label="Especialidade",
    formatar_colunas_como_hora=True,
    texto_template=".1f",
    largura_px=1000,
    altura_px=500,
)